# HOMEWORK 13

In this homework you are going to inspect the GTSRB (German Traffic Sign Recognition Benchmark) dataset. The dataset contains images of various classes of traffic signs used in Germany (and the whole EU). The objective of this homework is to go through the steps described below and to implement the necessary code.

At the end, as usual, there will be a couple of questions for you to answer. In addition, the last section of this homework is optional and, if you chose to do it, you'll earn extra point :-)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

### Step 0

Go to the GTSRB dataset official site ([link](https://benchmark.ini.rub.de/gtsrb_dataset.html)) to learn more about the dataset.

### Step 1

Download the dataset ([link](https://www.kaggle.com/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign)) and unzip it.

### Step 2

For this homework, you will be working with the training set. Check out the `Train.csv`, open it and see what it contains. Load the dataset and plot random samples.

In [ ]:
# Load the training labels
root = Path('../data/archive') # Path to the dataset location, e.g., '/data/janko/dataset/GTSRB'
data = pd.read_csv(os.path.join(root, 'Train.csv'))

# Number of training samples (amount of samples in data)
num_samples = len(data)

# Show random data samples
for ii in range(15):
    # Get random index
    idx = np.random.randint(0, num_samples)
    # Load image
    img = cv2.imread(os.path.join(root, data.iloc[idx]['Path']))
    # Convert image to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # TODO
    # Show image
    plt.subplot(3,5,ii+1), plt.imshow(img), plt.title(data.iloc[idx]['ClassId'])

### Step 3

Inspect the dataset by computing and plotting the per-class histogram.

In [ ]:
# Extract class identifiers
# Hint: Check the csv 
ids = data.ClassId

Compute the per class histogram. You can use any approach you want (e.g. `numpy`). It's also worth looking at the `Counter` function from the `collections` module ([link](https://docs.python.org/3/library/collections.html#collections.Counter)) ;-)

In [ ]:
from collections import Counter
hist = Counter(ids)

plt.bar(hist.keys(), hist.values()), plt.grid(True)
plt.xlabel('Traffic Sign ID'), plt.ylabel('Counts')

### Questions

Please answer the following questions:
* Do you consider the dataset to be balanced? If so, why? If not, why?
* Are there any classes that are (significantly) over-represented or under-represented?

Not at all, the idel is (almost) equal class representation. Here we have the range from 2250 (ids 2) examples to just 210 (ids 0, 19 ,37). That's a more than 10x difference!


* What would be the accuracy of a "classifier" that always predicts the most frequent class? What does it tell you about using accuracy as the only metric for this dataset?

2250 / num_samples = ~6% accuracy (0.0573), not critical but in more imbalanced datasets accuracy can easily be hacked to achieve 90%+ results, so we need to look for more balanced metrics like f1 to get the full picture.

* The images are `uint8` with values in [0, 255] and they have different sizes. What preprocessing would you apply before feeding them to a neural network (and why)?

Resize to some standard size and apply normalization to the inputs. Resizing is required because we have standard input size and the the whole math will become invalid if we will try to input more or less values that the number of input neurons. Normalization is useful for many reasons. For instance:

a. Training with various inputs. Here different number scales will have different impact on training with bigger numbers driving the process. Normalization solves than by providing a universal scale for all inputs.


b. Better gradient descent. Normalized data has better loss landscapes and prevents vanishing or exploding gradient. 

### Optional

Perform a further analysis on the dataset and draw some conclusion from it.

Hint 1: Unlike MNIST or CIFAR10, this dataset contains images with various spatial resolutions. Is there anything we can tell about the resolution distribution?

It's unequally distributed with median around 1800 pixels in total, but with example as big as 20000 pixels.

Hint 2: What about the brightness distribution? Are there classes there are significantly more bright than others?

Yes, lowest average brightness per class is around 40 (id 6) and highest -- over 120 (id 0; intensity difference of 80!).

These stats are the numeric proof for the need of resizing and data normalization discussed in previous point.

In [ ]:
data = data.assign(px_total = lambda df: df.Width * df.Height)
px_total_clipped = np.clip(data.px_total, a_min=None, a_max=5_000)
plt.hist(px_total_clipped, edgecolor='black')
plt.title('histogram of px_total')
plt.xlabel('count')
plt.ylabel('no_of_pixels')
plt.show()

In [ ]:
def calculate_brightness(relative_path):
    try:
        image_path = root / relative_path

        with Image.open(image_path) as img:
            grayscale = img.convert("L")
            return np.array(grayscale).mean()

    except (OSError, ValueError, TypeError):
        return np.nan

data["avg_brightness"] = data["Path"].apply(calculate_brightness)

In [ ]:
data.groupby(['ClassId']).agg({'avg_brightness':'mean'}).plot(kind='bar')